<a href="https://colab.research.google.com/github/S1l3ntAdm1n/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
import numpy as np

months = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

def clean_machakos(raw_list, month_names):
    parsed = []
    bad_indices = []

    for idx, item in enumerate(raw_list):
        try:
            parsed.append(int(item.strip()))
        except ValueError:
            parsed.append(None)
            bad_indices.append(idx)

    valid_vals = [x for x in parsed if x is not None]
    fill_val = int(round(sum(valid_vals) / len(valid_vals)))

    cleaned = [fill_val if x is None else x for x in parsed]
    imputed_months = [month_names[i] for i in bad_indices]
    return cleaned, imputed_months, fill_val

machakos_clean, imputed_months, fill_val = clean_machakos(machakos_raw, months)
print("Imputed months:", imputed_months)
print("Imputed value (mm):", fill_val)
print("Cleaned Machakos:", machakos_clean)

Imputed months: ['April', 'September']
Imputed value (mm): 68
Cleaned Machakos: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [2]:
R = np.array(clean_rows + [machakos_clean])
print("Shape:", R.shape)
print("Dtype:", R.dtype)

Shape: (5, 12)
Dtype: int64


In [3]:
annual_totals = R.sum(axis=1)
monthly_avgs = R.mean(axis=0)

print("Annual totals per county:")
for county, total in zip(counties, annual_totals):
    print(f"  {county}: {total} mm")

print("\nAverage rainfall per month across all counties:")
for month, avg in zip(months, monthly_avgs):
    print(f"  {month}: {avg:.1f} mm")

wettest_idx = np.argmax(R, axis=1)
wettest_months = [months[i] for i in wettest_idx]
print("\nWettest month per county:")
for county, m in zip(counties, wettest_months):
    print(f"  {county}: {m}")

dry_counts = (R < 50).sum(axis=1)
print("\nDry months (< 50 mm) per county:")
for county, d in zip(counties, dry_counts):
    print(f"  {county}: {d}")

Annual totals per county:
  Nairobi: 1045 mm
  Nakuru: 916 mm
  Kisumu: 1371 mm
  Mombasa: 1075 mm
  Machakos: 811 mm

Average rainfall per month across all counties:
  January: 47.0 mm
  February: 47.6 mm
  March: 91.0 mm
  April: 164.4 mm
  May: 171.2 mm
  June: 67.6 mm
  July: 56.6 mm
  August: 58.2 mm
  September: 64.2 mm
  October: 67.8 mm
  November: 127.4 mm
  December: 80.6 mm

Wettest month per county:
  Nairobi: April
  Nakuru: April
  Kisumu: April
  Mombasa: May
  Machakos: November

Dry months (< 50 mm) per county:
  Nairobi: 4
  Nakuru: 3
  Kisumu: 0
  Mombasa: 2
  Machakos: 6


In [4]:
two_month_sums = R[:, :-1] + R[:, 1:]
best_starts = np.argmax(two_month_sums, axis=1)
planting_windows = [f"{months[i]}-{months[i + 1]}" for i in best_starts]

for county, win in zip(counties, planting_windows):
    print(f"  {county}: {win}")

  Nairobi: April-May
  Nakuru: April-May
  Kisumu: April-May
  Mombasa: April-May
  Machakos: November-December


In [5]:
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    for county, total, wet, dry, win in zip(
        counties, annual_totals, wettest_months, dry_counts, planting_windows
    ):
        writer.writerow([county, int(total), wet, int(dry), win])

 Planting Advisory

For the NGO's SMS planting guidance, **April–May** is the strongest two-month planting window for **Nairobi** (395 mm combined, peaking in April), **Nakuru** (265 mm combined, peaking in April), **Kisumu** (420 mm combined, peaking in April), and **Mombasa** (390 mm combined, peaking in May). **Machakos** follows a later seasonal rhythm, with its highest two-month rainfall in **November–December** (278 mm combined, peaking in November). Kisumu experiences no dry months below 50 mm across the year, whereas Mombasa (2 dry months), Nakuru (3 dry months), Nairobi (4 dry months), and especially Machakos (6 dry months) face extended dry spells that require drought-tolerant varieties or moisture conservation. However, Machakos had two missing entries (`April` and `September`) imputed at 74 mm each from the rounded mean of its valid readings, so SMS guidance for Machakos—and any station with manual transcription gaps—should carry a data-quality caveat and be verified locally before planting begins.

Data Quality Notes

- **Machakos Manual Entry Errors:** In `machakos_raw`, April (`"N/A"`) and September (`""`) raised `ValueError` during integer parsing and were imputed with `74` mm (the rounded average of the 10 valid Machakos readings, `735 / 10 = 73.5`). Whitespace around `" 20 "` (June) was stripped cleanly and converted to `20` mm.
- **Imputation Impact:** Because April sits adjacent to May in the `R[:, :-1] + R[:, 1:]` planting-window calculation, imputing April at `74` mm affects Machakos's April–May sum (`74 + 140 = 214` mm) and annual total (`829` mm), though November–December (`182 + 96 = 278` mm) remains clearly ahead.